In [8]:
from pathlib import Path
from Bio import SeqIO

input_fasta = Path("H1N1_recurrent_origin.fasta")
output_fasta = Path("H1N1_recurrent_clean.fasta")

aa20 = set("ACDEFGHIKLMNPQRSTVWY")

seen = set()
records = []

total = 0
removed_non20aa = 0
removed_duplicate = 0
header_cleaned = 0

for record in SeqIO.parse(input_fasta, "fasta"):
    total += 1

    # 1. header清洗：按 | 分割，只保留第0部分
    old_id = record.id
    new_id = old_id.split("|", 1)[0]

    record.id = new_id
    record.name = new_id
    record.description = new_id

    # 2. 序列统一大写
    seq = str(record.seq).upper()
    record.seq = record.seq.__class__(seq)

    # 3. 去除含非20种标准氨基酸的序列
    if not set(seq) <= aa20:
        removed_non20aa += 1
        continue

    # 4. 按序列内容去重，保留第一次出现的序列
    if seq in seen:
        removed_duplicate += 1
        continue

    seen.add(seq)
    records.append(record)

SeqIO.write(records, output_fasta, "fasta")

print("清洗完成")
print(f"输入序列数: {total}")
print(f"因含非20种标准氨基酸被筛掉: {removed_non20aa}")
print(f"因序列重复被筛掉: {removed_duplicate}")
print(f"最终保留序列数: {len(records)}")
print(f"输出文件: {output_fasta}")

清洗完成
输入序列数: 2043
因含非20种标准氨基酸被筛掉: 485
因序列重复被筛掉: 579
最终保留序列数: 979
输出文件: H1N1_recurrent_clean.fasta
